#  G-02_조업이벤트.csv의 파일을 분석, 정리

| 컬럼 | 한글명 | 타입 | 단위 | 값 범위 | 결측 | 산출 |
| --- | --- | --- | --- | --- | --- | --- |
| EVT_ID | 이벤트번호 | 문자열 | — | 263종 ( EVT-2024-0001 ~ EVT-2024-0263 ) | 0 | |
| EVT_DT | 발생시각 | 문자열 | — | 2024-01-02 03:36 ~ 2024-12-31 20:15 | 0 | |
| EVT_TYPE | 이벤트유형 | 문자열 | — | 강종변경 / 원료배합변경 / 조업률변경 | 0 | 강종변경 / 원료배합변경 / 조업률변경 |
| PLANT_CD | 공장코드 | 문자열 | — | BF1 / BF2 / CR1 / SP1 / SP2 | 0 | 설비마스터와 잇는다 |
| BEF_VAL | 변경 전 값 | 문자열 | — | 40종 ( 100% ~ SPHC ) | 0 | 유형에 따라 의미가 다르다 |
| AFT_VAL | 변경 후 값 | 문자열 | — | 42종 ( 100% ~ SPHC ) | 0 | 유형에 따라 의미가 다르다 |

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Git\pdm-project")
data_path = project_root / "data" / "G-02_조업이벤트.csv"

df = pd.read_csv(data_path)

### 행 수와 컬럼 수

In [ ]:
print(f"행 수: {df.shape[0]}")
print(f"컬럼 수: {df.shape[1]}")

"""
행 수: 263
컬럼 수: 6
"""

### 컬럼 목록

In [ ]:
print("\n컬럼 목록:")
print(df.columns.tolist())

"""
컬럼 목록:
['EVT_ID', 'EVT_DT', 'EVT_TYPE', 'PLANT_CD', 'BEF_VAL', 'AFT_VAL']
"""

### 컬럼별 결측치 수

In [ ]:
print("\n컬럼별 결측치 수:")
display(df.isna().sum().to_frame(name="결측치 수"))

"""
컬럼별 결측치 수:
결측치 수
EVT_ID	0
EVT_DT	0
EVT_TYPE	0
PLANT_CD	0
BEF_VAL	0
AFT_VAL	0
"""

### 전체 행 중복 수

In [ ]:
print(f"\n완전히 중복된 행 수: {df.duplicated().sum()}")

"""
완전히 중복된 행 수: 0
"""

### EVT_ID 중복 수

In [ ]:
print(f"EVT_ID 중복 수: {df['EVT_ID'].duplicated().sum()}")

"""
EVT_ID 중복 수: 0
"""

### 강종변경만 추출한 데이터의 개수

In [ ]:
steel_grade_changes = df[df["EVT_TYPE"] == "강종변경"]

# 건수 출력 및 상위 5개 샘플만 확인
print(f"강종변경 건수: {len(steel_grade_changes)}건")
display(steel_grade_changes.head())

"""
EVT_ID	EVT_DT	EVT_TYPE	PLANT_CD	BEF_VAL	AFT_VAL
0	EVT-2024-0001	2024-01-02 03:36:00	강종변경	CR1	SPCC	SGCC
1	EVT-2024-0002	2024-01-05 23:46:00	강종변경	CR1	SGCC	SPFH590
2	EVT-2024-0003	2024-01-08 21:23:00	강종변경	CR1	SPFH590	SGCC
4	EVT-2024-0005	2024-01-11 22:34:00	강종변경	CR1	SGCC	SPCD
7	EVT-2024-0008	2024-01-15 19:01:00	강종변경	CR1	SPCD	SPFH590
"""

### EVT_TYPE 종류별 건수

In [ ]:
event_type_counts = (
    df["EVT_TYPE"]
    .value_counts()
    .reset_index()
)

event_type_counts.columns = ["EVT_TYPE", "건수"]

display(event_type_counts)

"""
EVT_TYPE	건수
0	강종변경	134
1	조업률변경	109
2	원료배합변경	20
"""

### PLANT_CD 종류별 건수

In [ ]:
plant_counts = (
    df["PLANT_CD"]
    .value_counts()
    .reset_index()
)

plant_counts.columns = ["PLANT_CD", "건수"]

display(plant_counts)

"""
	PLANT_CD	건수
0	CR1	134
1	BF1	47
2	BF2	41
3	SP1	23
4	SP2	18
"""

### PLANT_CD와 EVT_TYPE을 같이 놓고 어떤 설비에 어떤 이벤트가 있는지 표로 보기

In [ ]:
# PLANT_CD(행)와 EVT_TYPE(열)을 교차 집계하여 표 형태로 보기
plant_evt_crosstab = pd.crosstab(
    df["PLANT_CD"], 
    df["EVT_TYPE"], 
    margins=True, 
    margins_name="합계"
)

display(plant_evt_crosstab)

"""
EVT_TYPE	강종변경	원료배합변경	조업률변경	합계
PLANT_CD				
BF1	0	13	34	47
BF2	0	7	34	41
CR1	134	0	0	134
SP1	0	0	23	23
SP2	0	0	18	18
합계	134	20	109	263
"""

### 이벤트 종류별로 BEF_VAL, AFT_VAL에 어떤 값이 들어 있는지 몇 개씩 보기

In [ ]:
for evt_type, group in df.groupby("EVT_TYPE"):
    print(f"=== [EVT_TYPE: {evt_type}] ===")
    display(group[["BEF_VAL", "AFT_VAL"]].head(3))

"""
=== [EVT_TYPE: 강종변경] ===
BEF_VAL	AFT_VAL
0	SPCC	SGCC
1	SGCC	SPFH590
2	SPFH590	SGCC
=== [EVT_TYPE: 원료배합변경] ===
BEF_VAL	AFT_VAL
3	AUS60:BRA40	AUS55:BRA45
34	AUS45:BRA55	AUS50:BRA50
35	AUS55:BRA45	AUS45:BRA55
=== [EVT_TYPE: 조업률변경] ===
BEF_VAL	AFT_VAL
5	85%	92%
6	87%	93%
8	92%	99%
"""

### 월별 이벤트 건수 세기

In [ ]:
# 1. EVT_DT 컬럼을 datetime 타입으로 변환 (최초 1회 필수)
df["EVT_DT"] = pd.to_datetime(df["EVT_DT"])

# 2. 월 단위('YYYY-MM')로 집계
monthly_count = df["EVT_DT"].dt.to_period("M").value_counts().sort_index()

# 보기 좋게 데이터프레임으로 변환하여 출력
display(monthly_count.to_frame(name="이벤트건수"))

"""
이벤트건수
EVT_DT	
2024-01	19
2024-02	23
2024-03	26
2024-04	20
2024-05	24
2024-06	20
2024-07	20
2024-08	19
2024-09	23
2024-10	25
2024-11	23
2024-12	21
"""

## G-02 조업이벤트 분석 결과 요약

### 1. 데이터 기본 정보 및 품질 점검
| 항목 | 내용 | 비고 |
| :--- | :--- | :--- |
| **전체 데이터 규모** | 263행 × 6열 | `EVT_ID`, `EVT_DT`, `EVT_TYPE`, `PLANT_CD`, `BEF_VAL`, `AFT_VAL` |
| **결측치 및 중복** | 결측치 0건 / 중복행 0건 | 데이터 품질 양호 |
| **식별자(`EVT_ID`)** | 중복 없음 (263개 유일값) | 기본키(PK) 역할 적합 |
| **분석 기간** | 2024-01-02 ~ 2024-12-31 | 1년 치 월별 데이터 |

---

### 2. 설비(`PLANT_CD`)별 이벤트(`EVT_TYPE`) 발생 교차표
| 설비 (PLANT_CD) | 강종변경 | 원료배합변경 | 조업률변경 | 합계 | 주요 특징 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **BF1** (고로 1공장) | 0 | 13 | 34 | **47** | 원료 배합 및 조업률 변경 중심 |
| **BF2** (고로 2공장) | 0 | 7 | 34 | **41** | 원료 배합 및 조업률 변경 중심 |
| **CR1** (냉연 1공장) | 134 | 0 | 0 | **134** | 강종변경 이벤트만 발생 (전체 강종변경의 100%) |
| **SP1** (제강 1공장) | 0 | 0 | 23 | **23** | 조업률변경만 발생 |
| **SP2** (제강 2공장) | 0 | 0 | 18 | **18** | 조업률변경만 발생 |
| **합계** | **134** | **20** | **109** | **263** | **전체 263건** |

---

### 3. 이벤트 종류별 변경 전·후(`BEF_VAL` / `AFT_VAL`) 데이터 형식
| 이벤트 종류 (`EVT_TYPE`) | 변경 전 (`BEF_VAL`) 예시 | 변경 후 (`AFT_VAL`) 예시 | 주요 특징 요약 |
| :--- | :--- | :--- | :--- |
| **강종변경** | SPCC, SGCC, SPFH590 | SGCC, SPFH590, SPCD | 철강 제품 규격 및 강종 코드 |
| **원료배합변경** | AUS60:BRA40, AUS45:BRA55 | AUS55:BRA45, AUS50:BRA50 | 호주/브라질산 원료 비율 조합 |
| **조업률변경** | 85%, 87%, 92% | 92%, 93%, 99% | % 단위의 설비 운전 가동률 |

---

### 4. 2024년 월별 이벤트 발생 건수 추이
| 연월 (`EVT_DT`) | 이벤트 발생 건수 | 비고 |
| :---: | :---: | :--- |
| **2024-01** | 19건 | |
| **2024-02** | 23건 | |
| **2024-03** | 26건 | 연중 최다 발생 월 |
| **2024-04** | 20건 | |
| **2024-05** | 24건 | |
| **2024-06** | 20건 | |
| **2024-07** | 20건 | |
| **2024-08** | 19건 | |
| **2024-09** | 23건 | |
| **2024-10** | 25건 | |
| **2024-11** | 23건 | |
| **2024-12** | 21건 | |
| **전체 합계** | **263건** | **월평균 약 21.9건 발생** |

---

# 📊 8개 강종별(AFT_VAL) 이벤트 상세 및 요약


## 강종별 전체 현황 요약

| 강종명 | 발생 건수 | 전환 전 강종 종류 (`BEF_VAL`) | 최초 발생일 | 최근 발생일 |
| :---: | ---: | :--- | :---: | :---: |
| **DP590** | 12건 | SPCD, SPCE, SGCC, SPCC, SPHC, SPCEN, SPFH590 | 2024-04-21 18:18 | 2024-12-17 13:18 |
| **SGCC** | 15건 | SPCC, SPFH590, SPCD, SPCEN, SPCE, SPHC, DP590 | 2024-01-02 03:36 | 2024-11-26 00:25 |
| **SPCC** | 16건 | SPCE, SPFH590, SGCC, DP590, SPCEN, SPHC, SPCD | 2024-03-04 06:30 | 2024-12-26 16:22 |
| **SPCD** | 16건 | SGCC, SPCEN, SPHC, SPFH590, DP590, SPCE, SPCC | 2024-01-11 22:34 | 2024-12-23 18:36 |
| **SPCE** | 23건 | SGCC, SPHC, SPCC, DP590, SPCD, SPCEN | 2024-02-13 00:18 | 2024-12-28 14:57 |
| **SPCEN** | 15건 | SPFH590, SGCC, SPHC, SPCE, SPCD | 2024-01-24 06:01 | 2024-11-04 15:40 |
| **SPFH590** | 17건 | SGCC, SPCD, SPHC, SPCEN, SPCC, DP590, SPCE | 2024-01-05 23:46 | 2024-12-19 17:46 |
| **SPHC** | 20건 | SPFH590, SPCD, SPCC, DP590, SPCE, SPCEN | 2024-01-18 13:37 | 2024-12-31 20:15 |


### 📄 DP590 강종 변경 이벤트 목록
> **총 건수:** 12건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0080` | 2024-04-21 18:18 | SPCD | **DP590** |
| `EVT-2024-0085` | 2024-04-27 07:40 | SPCE | **DP590** |
| `EVT-2024-0117` | 2024-06-09 05:26 | SGCC | **DP590** |
| `EVT-2024-0121` | 2024-06-13 10:34 | SPCC | **DP590** |
| `EVT-2024-0136` | 2024-07-10 03:33 | SPHC | **DP590** |
| `EVT-2024-0144` | 2024-07-24 06:59 | SPCE | **DP590** |
| `EVT-2024-0166` | 2024-08-21 23:26 | SPCE | **DP590** |
| `EVT-2024-0169` | 2024-08-26 07:02 | SPCC | **DP590** |
| `EVT-2024-0185` | 2024-09-20 07:39 | SPCEN | **DP590** |
| `EVT-2024-0212` | 2024-10-21 05:41 | SPFH590 | **DP590** |
| `EVT-2024-0245` | 2024-12-02 23:56 | SPCD | **DP590** |
| `EVT-2024-0256` | 2024-12-17 13:18 | SPCE | **DP590** |


### 📄 SGCC 강종 변경 이벤트 목록
> **총 건수:** 15건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0001` | 2024-01-02 03:36 | SPCC | **SGCC** |
| `EVT-2024-0003` | 2024-01-08 21:23 | SPFH590 | **SGCC** |
| `EVT-2024-0019` | 2024-01-30 15:57 | SPCD | **SGCC** |
| `EVT-2024-0026` | 2024-02-11 22:58 | SPCEN | **SGCC** |
| `EVT-2024-0037` | 2024-02-22 15:42 | SPCD | **SGCC** |
| `EVT-2024-0054` | 2024-03-12 06:01 | SPCEN | **SGCC** |
| `EVT-2024-0061` | 2024-03-22 01:16 | SPCE | **SGCC** |
| `EVT-2024-0071` | 2024-04-06 22:39 | SPCEN | **SGCC** |
| `EVT-2024-0101` | 2024-05-19 06:30 | SPHC | **SGCC** |
| `EVT-2024-0116` | 2024-06-06 22:20 | SPCE | **SGCC** |
| `EVT-2024-0159` | 2024-08-10 15:25 | SPFH590 | **SGCC** |
| `EVT-2024-0176` | 2024-09-11 16:40 | SPHC | **SGCC** |
| `EVT-2024-0213` | 2024-10-23 23:36 | DP590 | **SGCC** |
| `EVT-2024-0232` | 2024-11-18 22:51 | SPCE | **SGCC** |
| `EVT-2024-0239` | 2024-11-26 00:25 | SPCE | **SGCC** |


### 📄 SPCC 강종 변경 이벤트 목록
> **총 건수:** 16건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0046` | 2024-03-04 06:30 | SPCE | **SPCC** |
| `EVT-2024-0057` | 2024-03-18 05:08 | SPFH590 | **SPCC** |
| `EVT-2024-0066` | 2024-03-25 06:46 | SGCC | **SPCC** |
| `EVT-2024-0074` | 2024-04-09 21:59 | SGCC | **SPCC** |
| `EVT-2024-0099` | 2024-05-13 06:57 | SPCE | **SPCC** |
| `EVT-2024-0120` | 2024-06-11 08:00 | DP590 | **SPCC** |
| `EVT-2024-0132` | 2024-06-29 14:00 | SPCEN | **SPCC** |
| `EVT-2024-0141` | 2024-07-19 04:48 | SPHC | **SPCC** |
| `EVT-2024-0168` | 2024-08-24 02:48 | DP590 | **SPCC** |
| `EVT-2024-0171` | 2024-08-30 11:02 | DP590 | **SPCC** |
| `EVT-2024-0178` | 2024-09-13 09:09 | SGCC | **SPCC** |
| `EVT-2024-0193` | 2024-09-27 23:52 | SPHC | **SPCC** |
| `EVT-2024-0201` | 2024-10-09 17:27 | SPCEN | **SPCC** |
| `EVT-2024-0207` | 2024-10-14 06:46 | SPCE | **SPCC** |
| `EVT-2024-0247` | 2024-12-08 02:31 | SPFH590 | **SPCC** |
| `EVT-2024-0261` | 2024-12-26 16:22 | SPCD | **SPCC** |


### 📄 SPCD 강종 변경 이벤트 목록
> **총 건수:** 16건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0005` | 2024-01-11 22:34 | SGCC | **SPCD** |
| `EVT-2024-0017` | 2024-01-27 00:03 | SPCEN | **SPCD** |
| `EVT-2024-0034` | 2024-02-19 03:51 | SPCEN | **SPCD** |
| `EVT-2024-0038` | 2024-02-25 15:59 | SGCC | **SPCD** |
| `EVT-2024-0068` | 2024-03-30 07:09 | SPHC | **SPCD** |
| `EVT-2024-0079` | 2024-04-19 06:38 | SPCEN | **SPCD** |
| `EVT-2024-0114` | 2024-06-02 07:12 | SPFH590 | **SPCD** |
| `EVT-2024-0137` | 2024-07-13 00:13 | DP590 | **SPCD** |
| `EVT-2024-0148` | 2024-07-27 22:39 | DP590 | **SPCD** |
| `EVT-2024-0152` | 2024-07-31 02:15 | SPCE | **SPCD** |
| `EVT-2024-0190` | 2024-09-22 23:47 | DP590 | **SPCD** |
| `EVT-2024-0194` | 2024-09-30 19:30 | SPCC | **SPCD** |
| `EVT-2024-0229` | 2024-11-13 07:08 | SPFH590 | **SPCD** |
| `EVT-2024-0234` | 2024-11-21 08:22 | SGCC | **SPCD** |
| `EVT-2024-0242` | 2024-11-29 04:50 | SGCC | **SPCD** |
| `EVT-2024-0259` | 2024-12-23 18:36 | SPFH590 | **SPCD** |


### 📄 SPCE 강종 변경 이벤트 목록
> **총 건수:** 23건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0028` | 2024-02-13 00:18 | SGCC | **SPCE** |
| `EVT-2024-0044` | 2024-03-02 10:09 | SPHC | **SPCE** |
| `EVT-2024-0059` | 2024-03-20 14:48 | SPCC | **SPCE** |
| `EVT-2024-0077` | 2024-04-13 23:16 | SPCC | **SPCE** |
| `EVT-2024-0083` | 2024-04-25 23:47 | DP590 | **SPCE** |
| `EVT-2024-0095` | 2024-05-10 22:32 | SPHC | **SPCE** |
| `EVT-2024-0102` | 2024-05-21 12:12 | SGCC | **SPCE** |
| `EVT-2024-0115` | 2024-06-04 22:23 | SPCD | **SPCE** |
| `EVT-2024-0125` | 2024-06-20 06:55 | SPCEN | **SPCE** |
| `EVT-2024-0133` | 2024-07-01 13:24 | SPCC | **SPCE** |
| `EVT-2024-0142` | 2024-07-21 17:27 | SPCC | **SPCE** |
| `EVT-2024-0151` | 2024-07-29 13:35 | SPCD | **SPCE** |
| `EVT-2024-0160` | 2024-08-12 02:21 | SGCC | **SPCE** |
| `EVT-2024-0164` | 2024-08-18 00:44 | SPHC | **SPCE** |
| `EVT-2024-0172` | 2024-09-01 23:38 | SPCC | **SPCE** |
| `EVT-2024-0203` | 2024-10-11 17:43 | SPCC | **SPCE** |
| `EVT-2024-0208` | 2024-10-16 09:14 | SPCC | **SPCE** |
| `EVT-2024-0218` | 2024-10-29 03:17 | SPCEN | **SPCE** |
| `EVT-2024-0223` | 2024-11-07 14:08 | SPCEN | **SPCE** |
| `EVT-2024-0230` | 2024-11-15 11:03 | SPCD | **SPCE** |
| `EVT-2024-0238` | 2024-11-24 07:31 | SPCD | **SPCE** |
| `EVT-2024-0254` | 2024-12-13 15:07 | SPHC | **SPCE** |
| `EVT-2024-0262` | 2024-12-28 14:57 | SPCC | **SPCE** |


### 📄 SPCEN 강종 변경 이벤트 목록
> **총 건수:** 15건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0015` | 2024-01-24 06:01 | SPFH590 | **SPCEN** |
| `EVT-2024-0020` | 2024-02-02 07:17 | SGCC | **SPCEN** |
| `EVT-2024-0025` | 2024-02-08 03:09 | SPHC | **SPCEN** |
| `EVT-2024-0030` | 2024-02-16 18:39 | SPCE | **SPCEN** |
| `EVT-2024-0053` | 2024-03-09 14:02 | SPFH590 | **SPCEN** |
| `EVT-2024-0069` | 2024-04-02 02:30 | SPCD | **SPCEN** |
| `EVT-2024-0078` | 2024-04-16 11:35 | SPCE | **SPCEN** |
| `EVT-2024-0110` | 2024-05-27 07:57 | SPFH590 | **SPCEN** |
| `EVT-2024-0123` | 2024-06-18 00:24 | SPHC | **SPCEN** |
| `EVT-2024-0131` | 2024-06-26 20:56 | SPHC | **SPCEN** |
| `EVT-2024-0134` | 2024-07-04 02:33 | SPCE | **SPCEN** |
| `EVT-2024-0182` | 2024-09-18 13:58 | SPHC | **SPCEN** |
| `EVT-2024-0198` | 2024-10-07 01:42 | SPHC | **SPCEN** |
| `EVT-2024-0216` | 2024-10-27 03:45 | SGCC | **SPCEN** |
| `EVT-2024-0220` | 2024-11-04 15:40 | SPHC | **SPCEN** |


### 📄 SPFH590 강종 변경 이벤트 목록
> **총 건수:** 17건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0002` | 2024-01-05 23:46 | SGCC | **SPFH590** |
| `EVT-2024-0008` | 2024-01-15 19:01 | SPCD | **SPFH590** |
| `EVT-2024-0013` | 2024-01-21 02:21 | SPHC | **SPFH590** |
| `EVT-2024-0023` | 2024-02-04 14:42 | SPCEN | **SPFH590** |
| `EVT-2024-0051` | 2024-03-07 21:22 | SPCC | **SPFH590** |
| `EVT-2024-0055` | 2024-03-15 23:08 | SGCC | **SPFH590** |
| `EVT-2024-0088` | 2024-04-30 23:25 | DP590 | **SPFH590** |
| `EVT-2024-0092` | 2024-05-06 01:59 | SPHC | **SPFH590** |
| `EVT-2024-0106` | 2024-05-24 01:18 | SPCE | **SPFH590** |
| `EVT-2024-0112` | 2024-05-29 21:18 | SPCEN | **SPFH590** |
| `EVT-2024-0157` | 2024-08-07 14:30 | SPHC | **SPFH590** |
| `EVT-2024-0173` | 2024-09-04 12:46 | SPCE | **SPFH590** |
| `EVT-2024-0196` | 2024-10-02 16:52 | SPCD | **SPFH590** |
| `EVT-2024-0210` | 2024-10-18 04:05 | SPCE | **SPFH590** |
| `EVT-2024-0226` | 2024-11-10 17:02 | SPCE | **SPFH590** |
| `EVT-2024-0246` | 2024-12-05 04:29 | DP590 | **SPFH590** |
| `EVT-2024-0258` | 2024-12-19 17:46 | DP590 | **SPFH590** |


### 📄 SPHC 강종 변경 이벤트 목록
> **총 건수:** 20건 | **담당 설비:** CR1

| 이벤트 ID | 발생 일시 (`EVT_DT`) | 변경 전 강종 (`BEF_VAL`) | 변경 후 강종 (`AFT_VAL`) |
| :---: | :---: | :---: | :---: |
| `EVT-2024-0010` | 2024-01-18 13:37 | SPFH590 | **SPHC** |
| `EVT-2024-0024` | 2024-02-06 06:51 | SPFH590 | **SPHC** |
| `EVT-2024-0041` | 2024-02-28 06:51 | SPCD | **SPHC** |
| `EVT-2024-0067` | 2024-03-28 06:12 | SPCC | **SPHC** |
| `EVT-2024-0091` | 2024-05-03 18:50 | SPFH590 | **SPHC** |
| `EVT-2024-0093` | 2024-05-08 19:45 | SPFH590 | **SPHC** |
| `EVT-2024-0100` | 2024-05-16 06:42 | SPCC | **SPHC** |
| `EVT-2024-0122` | 2024-06-15 01:04 | DP590 | **SPHC** |
| `EVT-2024-0130` | 2024-06-23 15:16 | SPCE | **SPHC** |
| `EVT-2024-0135` | 2024-07-07 16:57 | SPCEN | **SPHC** |
| `EVT-2024-0139` | 2024-07-16 06:51 | SPCD | **SPHC** |
| `EVT-2024-0155` | 2024-08-04 14:55 | SPCD | **SPHC** |
| `EVT-2024-0163` | 2024-08-14 12:05 | SPCE | **SPHC** |
| `EVT-2024-0174` | 2024-09-08 22:03 | SPFH590 | **SPHC** |
| `EVT-2024-0179` | 2024-09-15 06:25 | SPCC | **SPHC** |
| `EVT-2024-0191` | 2024-09-25 05:43 | SPCD | **SPHC** |
| `EVT-2024-0197` | 2024-10-04 22:58 | SPFH590 | **SPHC** |
| `EVT-2024-0219` | 2024-10-31 07:26 | SPCE | **SPHC** |
| `EVT-2024-0250` | 2024-12-10 05:00 | SPCC | **SPHC** |
| `EVT-2024-0263` | 2024-12-31 20:15 | SPCE | **SPHC** |
